# Lectures 1–2 Lab — Getting Started
## Your first Python, and what AI is doing to the economy

Welcome! This is the **first hands-on lab** of the course. By the end you will:

1. Have a working **Python + PyTorch** setup inside **VS Code** (using mirrors that are fast in China).
2. Run your **first Python program** and a small macro model.
3. See **why economists now study AI** — load real **Stanford AI Index** data and make three plots (**Lecture 1**).
4. Get a **first, intuitive taste of machine learning** — *learn a rule from labeled examples* (**Lecture 2**).

> **You only need VS Code to start** — everything else is installed below.
> **No Google account or VPN required**: we use mirrors that work well in mainland China.

*Philosophy:* you are the **Research Architect** — you specify, direct, and validate; Python is the construction crew.

# Part 0 — Set up your tools (VS Code, Python, PyTorch)

Do these **one-time** steps in your **Terminal** (macOS/Linux) or **Anaconda Prompt / PowerShell** (Windows). They use the **Tsinghua University (TUNA) mirrors**, which are fast and reliable in China.

### Step 1 — VS Code extensions
Open VS Code → Extensions (squares icon on the left) → install:
- **Python** (by Microsoft)
- **Jupyter** (by Microsoft)

### Step 2 — Install Python via Miniconda (China mirror)
Download the installer for your OS from the Tsinghua mirror and run it:

- Page: <https://mirrors.tuna.tsinghua.edu.cn/anaconda/miniconda/>
- **Windows:** newest `Miniconda3-latest-Windows-x86_64.exe`
- **macOS (Apple Silicon):** `Miniconda3-latest-MacOSX-arm64.sh` · **(Intel):** `...MacOSX-x86_64.sh`
- **Linux:** `Miniconda3-latest-Linux-x86_64.sh`, then `bash Miniconda3-latest-Linux-x86_64.sh`

### Step 3 — Point pip at the China mirror (one time)

```bash
pip config set global.index-url https://pypi.tuna.tsinghua.edu.cn/simple
```

### Step 4 — Create the course environment and install packages

```bash
conda create -n aiml2026 python=3.11 -y
conda activate aiml2026
pip install numpy pandas matplotlib scikit-learn torch
```

`torch` here is the **CPU build** — no GPU is required for this course.
If you cloned the course repo, you can instead run `pip install -r requirements.txt` from the `Labs/` folder.

### Step 5 — Open this notebook and pick the kernel
1. In VS Code: **File → Open Folder** → choose the course `Labs` folder.
2. Open `Lec01_02_Lab_Getting_Started.ipynb`.
3. Click **Select Kernel** (top-right) → **Python Environments** → **aiml2026**.
4. Run the next cell with **Shift+Enter** to check everything works.

> **Stuck on an error?** Copy the red error text and ask an AI assistant:
> *"I'm an economist new to Python on [Windows/macOS]. I got this error setting up VS Code + conda: &lt;paste&gt;. How do I fix it?"*

In [ ]:
# ✅ Run me first: check that the course tools are installed.
import sys, importlib
print("Python:", sys.version.split()[0])

def check(label, module):
    try:
        m = importlib.import_module(module)
        print(f"  OK   {label:13s} {getattr(m, '__version__', '')}")
        return True
    except Exception:
        print(f"  MISS {label:13s} (not installed)")
        return False

print("\nCourse packages:")
ready = True
for label, module in [("numpy","numpy"), ("pandas","pandas"),
                      ("matplotlib","matplotlib"), ("torch","torch")]:
    ready = check(label, module) and ready
check("scikit-learn", "sklearn")   # used in the Lecture 3 lab; optional here

import torch
device = torch.device("cuda" if torch.cuda.is_available()
                      else "mps" if getattr(torch.backends, "mps", None) and torch.backends.mps.is_available()
                      else "cpu")
print("\nPyTorch device:", device)
print("\n✅ Setup looks good — you are ready!" if ready
      else "\n⚠️  Install the missing packages (Part 0), then re-run this cell.")

# Part 1 — Your first Python

Run each cell with **Shift+Enter**. Change the numbers and re-run — you cannot break anything.

In [ ]:
# Variables and arithmetic
gdp    = 21.0      # trillions of USD
growth = 0.025     # 2.5% per year
print("GDP next year:", round(gdp * (1 + growth), 3), "trillion USD")

# A list and a for-loop: project GDP for 5 years
gdp_path = [gdp]
for year in range(5):
    gdp_path.append(gdp_path[-1] * (1 + growth))
print("5-year GDP path:", [round(x, 2) for x in gdp_path])

# A function
def compound(x0, rate, years):
    # value of x0 growing at `rate` for `years` periods
    return x0 * (1 + rate) ** years

print("GDP in 10 years:", round(compound(gdp, growth, 10), 2), "trillion USD")

### Computers and decimals: `0.1 + 0.2`

Computers store numbers in binary, so some decimals are slightly inexact. **Never test floats with `==`** — use `np.isclose` (or a tolerance). This matters when checking whether an algorithm has *converged*.

In [ ]:
import numpy as np
a = 0.1 + 0.2
print("0.1 + 0.2 =", a)
print("Exactly 0.3? ", a == 0.3)
print("Close to 0.3?", np.isclose(a, 0.3))

### Your first macro model: the Solow growth model

Capital per worker evolves as $k_{t+1} = s\,k_t^{\alpha} + (1-\delta)\,k_t$ and converges to the steady state $k^*=(s/\delta)^{1/(1-\alpha)}$. Let's simulate and plot it.

In [ ]:
%matplotlib inline
import numpy as np
import matplotlib.pyplot as plt

# Parameters
alpha = 0.33   # capital share
s     = 0.20   # saving rate
delta = 0.05   # depreciation
T     = 200    # periods
k     = 0.1    # initial capital

k_path = [k]
for t in range(T):
    k = s * k**alpha + (1 - delta) * k      # law of motion
    k_path.append(k)

k_star = (s / delta) ** (1 / (1 - alpha))   # analytical steady state

plt.figure(figsize=(9, 4))
plt.plot(k_path, lw=2, label="Simulated $k_t$")
plt.axhline(k_star, color="red", ls="--", label=f"Steady state $k^*={k_star:.2f}$")
plt.xlabel("Year"); plt.ylabel("Capital per worker")
plt.title("Solow growth model — convergence to steady state")
plt.legend(); plt.grid(alpha=0.3); plt.show()

print(f"Final k = {k_path[-1]:.4f},  steady state k* = {k_star:.4f}")

### Doing it faster with NumPy

Loops are fine for small problems, but **NumPy** computes on whole arrays at once ("vectorization") — the workhorse of quantitative macro.

In [ ]:
k_grid = np.linspace(0.1, 10, 100)
y_grid = k_grid ** alpha            # vectorized: no loop needed
plt.figure(figsize=(9, 4))
plt.plot(k_grid, y_grid, lw=2)
plt.xlabel("Capital $k$"); plt.ylabel(r"Output $y=k^{\alpha}$")
plt.title("Production function (computed with NumPy vectorization)")
plt.grid(alpha=0.3); plt.show()

> **✏️ Your turn:** in the Solow cell, change the saving rate `s` from `0.20` to `0.30` and re-run. Does steady-state capital $k^*$ rise or fall? Why?

# Part 2 — The Economics of AI in three pictures

**Lecture 1** framed AI as a new **general-purpose technology** and as **"cognitive capital."** Three forces drive its economic impact: AI is becoming **more capable**, **cheaper to run**, and **adopted faster** than earlier technologies. Let's *see* each one, using data from the **Stanford AI Index** (bundled in `data/`, with citations).

In [ ]:
import os
import pandas as pd

def find_data(filename):
    # Locate a bundled data file whether you run from Labs/ or the repo root.
    for path in [os.path.join("data", filename),
                 filename,
                 os.path.join("FullCourse_10Lecs", "Labs", "data", filename)]:
        if os.path.exists(path):
            return path
    raise FileNotFoundError(f"Could not find {filename}: keep the 'data/' folder next to this notebook.")

perf  = pd.read_csv(find_data("ai_performance_vs_human.csv"))
cost  = pd.read_csv(find_data("inference_cost.csv"))
adopt = pd.read_csv(find_data("adoption_rates.csv"))

print("AI vs human (MMLU):"); print(perf.to_string(index=False)); print()
print("Inference cost:");     print(cost.to_string(index=False)); print()
print("Adoption speed:");     print(adopt.to_string(index=False))

### Plot 1 — AI frontier capability vs. human experts  *(worked example)*

**MMLU** tests knowledge across 57 subjects. The dashed line is the **human-expert** score (89.8%). Watch the best models close — and cross — the gap. This is the **"capability"** force from **Lecture 1** ("The Frontier Is Rising Fast").

In [ ]:
plt.figure(figsize=(9, 5))
plt.plot(perf["Year"], perf["AI_Score"], "o-", lw=2, label="Best AI model (MMLU)")
plt.axhline(perf["Human_Expert_Baseline"].iloc[0], color="red", ls="--",
            label="Human expert baseline (89.8%)")
for _, r in perf.iterrows():
    plt.annotate(r["Model"], (r["Year"], r["AI_Score"]),
                 textcoords="offset points", xytext=(0, 8), fontsize=8, ha="center")
plt.xlabel("Year"); plt.ylabel("MMLU accuracy (%)")
plt.title("Plot 1 — AI frontier capability vs. human experts")
plt.ylim(30, 100); plt.legend(loc="lower right"); plt.grid(alpha=0.3); plt.show()

### (Optional) Extract the latest data *live* from Our World in Data

Our World in Data is reachable from China and publishes a clean CSV of generative-AI adoption by country. This cell is **optional** and safely skips itself if you are offline.

In [ ]:
# OPTIONAL live data extraction (safe to skip offline).
import io, urllib.request
OWID_URL = ("https://ourworldindata.org/grapher/estimated-share-people-generative-ai.csv"
            "?v=1&csvType=full&useColumnShortNames=false")
try:
    with urllib.request.urlopen(OWID_URL, timeout=8) as resp:
        owid = pd.read_csv(io.BytesIO(resp.read()))
    print("Live OWID data fetched:", owid.shape)
    print(owid.head())
except Exception as e:
    owid = None
    print("(Skipped live fetch — continuing with bundled data.)  Reason:", type(e).__name__)

### Plot 2 — The collapsing cost of inference   ✏️ *your turn*

Running a model good enough to match **GPT-3.5** fell from about **\$20** to **\$0.07** per million tokens in ~18 months (≈ **280× cheaper**). A basic plot is provided — **improve it**: log scale, axis labels, a title, and an annotation. As the **price of cognition** falls, more tasks become worth automating — the **"cheaper"** force from **Lecture 1**.

In [ ]:
# ✏️ YOUR TURN (1): improve this plot. A basic version runs so you can see the data.
# Tasks:  - put the y-axis on a LOG scale:  plt.yscale("log")
#         - add x/y labels and a title
#         - annotate the ~280x drop
# A polished solution is in the Appendix at the end.
plt.figure(figsize=(9, 5))
plt.plot(cost["Date"], cost["USD_per_Million_Tokens"], "o-")
plt.xticks(rotation=45)
# TODO: yscale("log"); labels; title; annotation
plt.tight_layout(); plt.show()

### Plot 3 — AI adoption vs. earlier technologies   ✏️ *your turn*

How long did each technology take to reach ~50% adoption? The PC took ~15 years, the internet ~7, generative AI ~3. Turn the bundled numbers into a clear bar chart — **sort the bars** and **highlight Generative AI**. This is the **"faster diffusion"** force from **Lecture 1**.

> **Tie-in (Lecture 1 — the GPT thesis).** True *general-purpose technologies* — steam, electricity, the computer — reshaped the economy, but only over **decades** of complementary investment. AI is diffusing far faster. Whether that makes it a genuine GPT (and not another over-hyped "next big thing") is an **open, empirical question** — exactly the kind economists are here to answer.

In [ ]:
# ✏️ YOUR TURN (2): improve this bar chart (sort bars; highlight Generative AI; add labels/title).
plt.figure(figsize=(8, 5))
plt.bar(adopt["Technology"], adopt["Years_to_50pct_Adoption"])
# TODO: sort by years; color the Generative AI bar; ylabel + title
plt.tight_layout(); plt.show()

### Bonus — from adoption to *automation*: the S-curve

Lecture 1 argued AI spreads **task by task**: as inference gets **cheaper** and **more capable**, each task crosses a break-even point and gets automated — **easy, high-volume tasks first**, hard or rare tasks much later. That traces out an **S-curve** of automation. Here it is as a tiny simulation (no data needed) — the same picture as the *"Adoption Follows an S-Curve"* slide.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

t = np.linspace(0, 10, 200)                 # "time" = AI getting cheaper + more capable
easy = 100 / (1 + np.exp(-1.3 * (t - 4)))   # high-scale, low-cognition tasks: automate early
hard =  65 / (1 + np.exp(-0.7 * (t - 7)))   # low-scale, high-cognition tasks: lag, plateau lower

plt.figure(figsize=(9, 4))
plt.plot(t, easy, lw=2, label="high-scale, low-cognition tasks")
plt.plot(t, hard, lw=2, ls="--", label="low-scale, high-cognition tasks")
plt.xlabel("Time  (as inference gets cheaper and more capable)")
plt.ylabel("% of tasks automated")
plt.title("Adoption follows an S-curve — easy, high-volume tasks first")
plt.ylim(-5, 105); plt.legend(); plt.grid(alpha=0.3); plt.tight_layout(); plt.show()

# ✏️ Your turn: make the "hard" tasks catch up faster — raise its steepness 0.7 toward 1.3.

### Why this matters, and make it your own

Together these pictures say: AI is getting **more capable**, **cheaper**, and **spreading faster** than previous general-purpose technologies — which is why **Lecture 1** studies it as a new kind of (cognitive) capital.

But remember the caution from the slides: **"GPT status is an *ex-post* verdict, not a press release."** Plenty of technologies once looked transformational and fizzled. The frontier is moving on a timescale of *months*, so treat every number here as a **current snapshot**, not a settled fact.

**Make it your own:**
- Run the optional OWID cell and plot generative-AI adoption for a few countries — where does **China** sit?
- Add the newest model you know to `data/ai_performance_vs_human.csv` and re-run Plot 1.
- In Plot 2, compute the *annual* cost-reduction factor.

# Part 3 — What is AI? A first taste of machine learning

**Lecture 2 ("What is AI?")** surveys the technique families. The workhorse is **supervised learning**, and the idea is simple:

> **Show the model many examples — each an input paired with the correct answer — and it learns a rule that generalizes to new inputs.**

- **Regression** predicts a **number** (a wage, GDP growth, a price).
- **Classification** predicts a **category** (recession or not? default or not?).

Let's *do* it. We'll give a model 60 labeled examples of *(years of education → hourly wage)* and let it learn the rule — then use it to predict. No formulas to memorize: this is **"curve-fitting taken to its limit."**

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# 60 labeled examples: input x = years of education, label y = hourly wage (with noise)
rng  = np.random.default_rng(0)
edu  = rng.uniform(8, 20, 60)
wage = 5 + 2.3 * edu + rng.normal(0, 4, edu.size)

# "Learn the rule" from the labeled examples:  wage ~ a + b * education
b, a = np.polyfit(edu, wage, 1)
print(f"Learned rule:   wage ~ {a:.1f} + {b:.2f} * years_of_education")

# Use the learned rule to PREDICT for someone the model never saw:
x_new = 16
print(f"Prediction at {x_new} years of schooling:   ${a + b * x_new:.1f} / hour")

xs = np.linspace(8, 20, 100)
plt.figure(figsize=(9, 4))
plt.scatter(edu, wage, s=22, alpha=0.7, label="labeled examples (data)")
plt.plot(xs, a + b * xs, "C3", lw=2, label="learned rule")
plt.xlabel("Years of education  (input $x$)"); plt.ylabel("Hourly wage  (label $y$)")
plt.title("Supervised learning: learn a rule from labeled examples")
plt.legend(); plt.grid(alpha=0.3); plt.tight_layout(); plt.show()

# ✏️ Your turn: predict at x_new = 12 or 20; or change the true slope 2.3 and re-run.

### How does it "learn"? *predict → measure error → nudge → repeat*

`np.polyfit` solved for the best line in one shot. A **neural network** finds it a different way — the loop Lecture 2 described:

1. **predict** with the current guess → 2. **measure the error** → 3. **nudge** the parameters in the direction that lowers it → 4. **repeat**.

Here is that loop in a few lines of NumPy. Watch the error fall. *(In Lecture 3 you'll run the same idea in **PyTorch**, which does the calculus for you and scales it to millions of parameters.)*

In [ ]:
# Same line fit, but learned by gradient descent — the engine inside every neural network.
xm, xstd = edu.mean(), edu.std()
x = (edu - xm) / xstd            # standardize the input so the learning steps are stable
y = wage

slope, intercept, lr = 0.0, 0.0, 0.1     # start from a blank guess
for step in range(400):
    pred  = intercept + slope * x        # 1. predict with the current guess
    error = pred - y                     # 2. measure the error
    intercept -= lr * error.mean()       # 3. nudge the parameters downhill...
    slope     -= lr * (error * x).mean() #    ...in the direction that lowers the error
    if step % 100 == 0:                  # 4. repeat
        print(f"step {step:3d}   mean squared error = {(error**2).mean():7.2f}")

x16 = (16 - xm) / xstd
print("\nLearned by 'predict -> measure error -> nudge -> repeat'.")
print(f"Prediction at 16 years: ${intercept + slope * x16:.1f}/hour  "
      f"(one-shot polyfit said ${a + b * 16:.1f}) -- same answer, found by learning.")

### A caution from Lecture 2: a good prediction ≠ a good decision

That worked because wages really do track education and we had clean, labeled data. **Real prediction problems are often far harder — and "accurate" is not the same as "right":**

- **COMPAS** (criminal-risk scoring) was only ~64% accurate, flagged Black defendants who did *not* reoffend at **twice** the rate of White defendants, and predicted **arrests**, not crime — the label it *claimed* to predict was not the label it *measured*.
- The **Fragile Families Challenge** gave 160 teams ~10,000 features on ~4,000 children; the best AI **barely beat a coin flip** at predicting life outcomes — some futures simply are not in the data.

This is why **you are the Research Architect**: before trusting any model, ask **what task? what data? what claim?** — and always validate on data the model has **not** seen. We return to this discipline throughout the course (Lec. 7, Lec. 9).

---
## Appendix — Reference solutions

Try the ✏️ tasks yourself first, then compare.

In [ ]:
# --- Solution: Plot 2 (inference cost on a log scale) ---
fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(cost["Date"], cost["USD_per_Million_Tokens"], "o-", color="C3", lw=2)
ax.set_yscale("log")
ax.set_xlabel("Date"); ax.set_ylabel("USD per million tokens (log scale)")
ax.set_title("Plot 2 — Cost to reach GPT-3.5-level quality")
ax.annotate("≈ 280× cheaper in ~18 months", xy=(0.5, 0.6),
            xycoords="axes fraction", ha="center", fontsize=11)
plt.setp(ax.get_xticklabels(), rotation=45, ha="right")
ax.grid(alpha=0.3, which="both"); plt.tight_layout(); plt.show()

In [ ]:
# --- Solution: Plot 3 (adoption speed) ---
d = adopt.sort_values("Years_to_50pct_Adoption", ascending=False)
colors = ["C2" if t == "Generative AI" else "C7" for t in d["Technology"]]
plt.figure(figsize=(8, 5))
plt.bar(d["Technology"], d["Years_to_50pct_Adoption"], color=colors)
plt.ylabel("Years to reach ~50% adoption")
plt.title("Plot 3 — How fast did each technology spread?")
for i, v in enumerate(d["Years_to_50pct_Adoption"]):
    plt.text(i, v + 0.2, str(v), ha="center")
plt.tight_layout(); plt.show()

---
## Wrap-up

You installed Python + PyTorch in VS Code, ran your first Python and a Solow model, visualized three economic facts about AI (**Lecture 1**), and trained your **first model to learn a rule from data** (**Lecture 2**).

The two lectures, in one lab: **Lecture 1** asks *why economists should care about AI* — a cheap, fast-improving, fast-diffusing new form of capital; **Lecture 2** asks *what AI actually is* — techniques that **learn rules from data**, powerful but **bounded** (recall the COMPAS and Fragile Families cautions).

**Next:** `Lec03_Lab_ML_Basics.ipynb` — scale today's tiny "predict → measure error → nudge → repeat" loop into a real neural network in PyTorch. Your environment is already set up, so you can dive straight in.

*Data sources:* Stanford AI Index 2025 (hai.stanford.edu), Epoch AI, and Our World in Data. See `data/*.csv` and `README.md` for citations.